# 00 · Real IBM QPU
Run **at the start of the talk** (and once in the morning as plan B). The queue can take minutes,
so we send now and collect the results during the demos. Same circuits as in demos 1 and 2, written in Qiskit.

In [1]:
import json, os
from datetime import date
from dotenv import load_dotenv
from qiskit import QuantumCircuit
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

load_dotenv()
service = QiskitRuntimeService(channel="ibm_quantum_platform",
                               token=os.environ["IBM_QUANTUM_PLATFORM_API_KEY"])
backend = service.least_busy(operational=True, simulator=False)
print("backend:", backend.name)

qiskit_runtime_service._discover_account:WARNING:2026-11-24 12:02:29,459: Loading account with the given token. A saved account will not be used.
qiskit_runtime_service.__init__:WARNING:2026-11-24 12:02:34,288: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-11-24 12:02:34,851: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-11-24 12:02:37,147: Using instance: open-instance, plan: open


backend: ibm_fez


In [2]:
# Demo 1: Bell state
bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

# Demo 2: Grover, 3 qubits, secret 101, 2 iterations
def ccz(qc):
    qc.h(2); qc.ccx(0, 1, 2); qc.h(2)

grover = QuantumCircuit(3)
grover.h(range(3))
for _ in range(2):
    grover.x(1); ccz(grover); grover.x(1)                                  # oracle: flip the sign of 101
    grover.h(range(3)); grover.x(range(3)); ccz(grover); grover.x(range(3)); grover.h(range(3))  # diffusion
grover.measure_all()

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa = [pm.run(bell), pm.run(grover)]
print("two-qubit gates after transpiling:", [c.count_ops().get("cz", 0) + c.count_ops().get("ecr", 0) for c in isa])

two-qubit gates after transpiling: [1, 45]


In [3]:
job = Sampler(mode=backend).run(isa, shots=1000)
json.dump({"job_id": job.job_id(), "backend": backend.name}, open("ibm_job.json", "w"))
print("sent:", job.job_id())

sent: dc2mtluq4uuc73a5jbmg


### Later: collect the results

In [4]:
saved = json.load(open("ibm_job.json"))
job = service.job(saved["job_id"])
print("status:", job.status())

result = job.result()
results = {name: {"counts": result[i].data.meas.get_counts(),
                  "backend": saved["backend"],
                  "date": date.today().isoformat()}
           for i, name in enumerate(["bell", "grover"])}
json.dump(results, open("ibm_results.json", "w"), indent=1)
print(results["bell"]["counts"])
print("QPU time used:", job.usage(), "s")

status: RUNNING
{'00': 489, '11': 466, '01': 21, '10': 24}
QPU time used: 0 s
